In [1]:
# ==================================================================================
# REGENERATED SCRIPT: TRANSFORMER ARCHITECTURE (V5)
# - Base Architecture: GCN + Transformer Encoder/Decoder (as requested initially)
# - Dataset Updates: Removed Category Metadata (12 Colors, 90 Objects).
# ==================================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import statistics
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import os
import evaluate as hf_evaluate 

# Ensure correct SciPy compatibility check is handled (often outside the script scope)
# /home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)

# ==================================================================================
# --- CONSTANTS & GLOBAL SETUP (UPDATED FOR NEW DATASET) ---
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
MODEL_SAVE_PATH = 'eeg-meta-text-transformer-v5-model.pt'
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32 # Increased batch size back to original for Transformer memory efficiency

# NEW DATASET CONSTANTS
NUM_COLORS = 12
NUM_OBJECTS = 90
# NUM_CATEGORIES = 0 # Removed

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
D_MODEL = 256 # Transformer hidden dimension

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
NUM_LAYERS = 4
NUM_HEADS = 8
D_FF = 1024
DROPOUT = 0.1
EPOCHS = 50

# Loss Weights (Adjusted based on previous successful training attempts)
OBJECT_LOSS_WEIGHT = 0.5 
COLOR_LOSS_WEIGHT = 0.2 

print(f"Using device: {device}")
print(f"Metadata config: {NUM_COLORS} colors, {NUM_OBJECTS} objects (NO categories)")
print(f"Loss weights: Object={OBJECT_LOSS_WEIGHT}, Color={COLOR_LOSS_WEIGHT}")

# ==================================================================================
# --- TRANSFORMER CORE COMPONENTS (UNCHANGED) ---
# ==================================================================================

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=DROPOUT):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        batch_size = query.size(0)
        
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Metadata Conditional Bias (Semantic Gating)
        self.meta_gate = nn.Sequential(
            nn.Linear(d_meta, d_model),
            nn.Sigmoid()
        )
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask, meta_features):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        
        # Apply Metadata Conditional Bias (Semantic Gating)
        gate = self.meta_gate(meta_features).unsqueeze(1) # (B, 1, D_model)
        x = x * gate 
        
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# ==================================================================================
# --- METADATA ENCODER (UPDATED: NO CATEGORY) ---
# ==================================================================================

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, 
                 color_emb_dim=16, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, object_feature_dim)
        )
        
        # Metadata input is [Color_ID, Object_Multi_Hot...]
        self.output_dim = color_emb_dim + object_feature_dim

    def forward(self, metadata):
        # metadata[:, 0] is Color ID (long)
        # metadata[:, 1:] is Object Multi-hot/soft vector (float)
        color_ids = metadata[:, 0].long()
        object_features_raw = metadata[:, 1:].float()
        
        color_vec = self.color_embedding(color_ids)
        object_vec = self.object_processor(object_features_raw)

        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features

# ==================================================================================
# --- SPATIO-TEMPORAL EEG ENCODER (UNCHANGED) ---
# ==================================================================================

class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=D_MODEL, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.spatial_dropout = nn.Dropout(dropout)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # --- Spatial Processing (GCN) ---
        batch_edge_index, batch_edge_attr = self._prepare_gcn_input(batch_size, num_timesteps, edge_index, edge_attr)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels) # (B*T, C)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.spatial_dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        
        x = x.reshape(batch_size, num_timesteps, self.d_model)
        
        # --- Transformer Input Preparation ---
        cls_token = self.cls_token.repeat(batch_size, 1, 1) # (B, 1, D_model)
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        x = self.pos_encoding(x)

        # --- Temporal Processing (Transformer Encoder Stack) ---
        for layer in self.transformer_layers:
            x = layer(x)
        
        return self.layer_norm(x)

    def _prepare_gcn_input(self, batch_size, num_timesteps, edge_index, edge_attr):        
        # Adapted for correctness from the previous version
        num_channels = self.num_channels
        base_edge_index = edge_index
        num_edges = base_edge_index.size(1)
        
        batch_size_indices = torch.arange(batch_size, device=edge_index.device)
        offsets = batch_size_indices * num_channels
        offsets_repeated = offsets.repeat_interleave(num_edges)
        
        batch_edge_index = base_edge_index.repeat(1, batch_size) + offsets_repeated
        batch_edge_attr = edge_attr.repeat(batch_size)
        return batch_edge_index, batch_edge_attr

# ==================================================================================
# --- TRANSFORMER DECODER (UNCHANGED) ---
# ==================================================================================

class DecoderTF(nn.Module):
    def __init__(self, vocab_size, emb_dim, d_model, num_layers, num_heads, d_ff, pad_id, dropout, d_meta):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(emb_dim)
        self.input_projection = nn.Linear(emb_dim, d_model) 
        
        decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, d_meta)
        self.transformer_layers = get_clones(decoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, target_text_ids, memory, memory_mask, meta_features):
        tgt_embed = self.embedding(target_text_ids)
        x = self.pos_encoding(tgt_embed)
        x = self.input_projection(x)

        tgt_seq_len = target_text_ids.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(x.device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.transformer_layers:
            x = layer(x, memory, memory_mask, tgt_mask, meta_features)
        
        x = self.layer_norm(x)
        return self.fc_out(x)

# ==================================================================================
# --- SEQ2SEQ TRANSFORMER (UPDATED: NO CATEGORY) ---
# ==================================================================================

class Seq2SeqTF(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, 
                 d_model=D_MODEL, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, d_ff=D_FF, pad_id=PAD_ID, dropout=DROPOUT, 
                 color_emb_dim=16, object_feature_dim=128):
        super().__init__()
        
        self.encoder = SpatioTemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.meta_encoder = MetadataEncoder(
            num_colors, num_objects, color_emb_dim, object_feature_dim
        )
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = DecoderTF(
            text_vocab_size, d_model, d_model, num_layers, num_heads, d_ff, pad_id, dropout, meta_features_dim
        )
        
        # Meta head now predicts only color and object
        self.meta_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects
        self.pad_id = pad_id

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, meta_teacher_forcing_ratio=1.0):
        eeg_features = self.encoder(eeg, edge_index, edge_attr) # (B, T+1, D_model)
        
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        meta_preds = self.meta_head(cls_token_feature)
        
        # --- Metadata Conditioning (Scheduled Sampling) ---
        use_true_meta = random.random() < meta_teacher_forcing_ratio
        
        if use_true_meta:
            # GT Metadata input: [Color_ID, Object_Multi_Hot...]
            meta_features = self.meta_encoder(metadata)
        else:
            with torch.no_grad():
                # Predicted Metadata input: [Color_ID (argmax), Object_Soft_Sigmoid]
                pred_color_logits = meta_preds[:, :self.num_colors]
                pred_object_logits = meta_preds[:, self.num_colors:]
                
                pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
                
                # Use soft predictions (sigmoid) for the multi-label object features 
                pred_object_soft = torch.sigmoid(pred_object_logits) 
                
                predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_soft], dim=1)
            meta_features = self.meta_encoder(predicted_meta_vector)
        
        # Decoder Memory
        decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
        memory_mask = None 
        
        text_logits = self.decoder(
            target_text[:, :-1], # Target input (shifted right)
            decoder_memory, 
            memory_mask, 
            meta_features
        )
        
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]
        
        return text_logits, pred_color, pred_object

# ==================================================================================
# --- DATASET AND GRANGER UTILS (UNCHANGED/SIMPLIFIED) ---
# ==================================================================================

class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j: continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            try:
                # maxlag=5 is used for the test
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                # We use the F-test result from the 5th lag
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    if edge_attr is None:
        edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# ==================================================================================
# --- TRAINING AND EVALUATION FUNCTIONS (UPDATED: NO CATEGORY) ---
# ==================================================================================

def train_one_epoch_tf(model, loader, optimizer, text_criterion, color_criterion, object_criterion, 
                       granger_edge_index, granger_edge_attr, color_loss_weight, object_loss_weight, 
                       meta_teacher_forcing_ratio):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training TF", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        # meta_b[:, 0] is Color ID (GT)
        # meta_b[:, 1:] is Object Multi-Hot (GT)
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, 
            meta_teacher_forcing_ratio=meta_teacher_forcing_ratio
        )
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        object_loss = object_criterion(pred_object, meta_b[:, 1:].float())
        
        loss = text_loss + (color_loss_weight * color_loss) + (object_loss_weight * object_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(
            text_loss=text_loss.item(), 
            color_loss=color_loss.item(), 
            object_loss=object_loss.item()
        )
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_tf(model, loader, text_criterion, color_criterion, object_criterion, 
                granger_edge_index, granger_edge_attr, color_loss_weight, object_loss_weight):
    model.eval()
    total_loss = 0.0
    
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        # CRITICAL: Use meta_teacher_forcing_ratio=0.0 for HONEST evaluation
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, 
            meta_teacher_forcing_ratio=0.0
        )
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        object_loss = object_criterion(pred_object, meta_b[:, 1:].float())
        
        loss = text_loss + (color_loss_weight * color_loss) + (object_loss_weight * object_loss)
        total_loss += loss.item()
        
    return total_loss / len(loader)



/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cuda
Metadata config: 12 colors, 90 objects (NO categories)
Loss weights: Object=0.5, Color=0.2


In [2]:
# ==================================================================================
# --- MAIN EXECUTION BLOCK ---
# ==================================================================================

if __name__ == "__main__":
    
    # --- Checkpoint Cleanup ---
    try:
        if os.path.exists(MODEL_SAVE_PATH):
            os.remove(MODEL_SAVE_PATH)
            print(f"Removed old checkpoint: '{MODEL_SAVE_PATH}'")
    except Exception as e:
        print(f"Could not remove old checkpoint: {e}")

    # --- 1. Setting up Data and Static Graph ---
    g = torch.Generator().manual_seed(42)
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val   = int(N * VAL_PCT)
    n_test  = N - n_train - n_val
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    print(f"\n--- 1. Setting up Data and Static Graph ---")
    print(f"Data loaders created: Train={n_train}, Val={n_val}, Test={n_test}")

    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]

        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels, fill_value=1.0
        )
        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Static Granger Graph ready on {device}. Edges: {granger_edge_index.shape[1]}")
    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Cannot continue without it.")
        exit()

    # --- 2. Model and Loss Initialization ---
    model = Seq2SeqTF(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS, num_objects=NUM_OBJECTS,
        d_model=D_MODEL, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, d_ff=D_FF, pad_id=PAD_ID, dropout=DROPOUT
    ).to(device)

    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.CrossEntropyLoss()
    object_criterion = nn.BCEWithLogitsLoss()

    # --- 3. Training Loop Setup ---
    new_optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)
    best_val_loss = float('inf')
    
    meta_tf_schedule = np.linspace(1.0, 0.5, 20) # Schedule metadata teacher forcing

    print(f"\n--- 2. Starting Transformer Training (V5) ---")
    print(f"Total Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Validation uses Honest Evaluation (Meta TF Ratio = 0.0)")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        
        # Get scheduled metadata teacher forcing ratio
        current_meta_tf_ratio = meta_tf_schedule[epoch - 1] if epoch - 1 < len(meta_tf_schedule) else meta_tf_schedule[-1]
        
        train_loss = train_one_epoch_tf(
            model, train_loader, new_optimizer, text_criterion, color_criterion, object_criterion, 
            granger_edge_index, granger_edge_attr, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT, 
            meta_teacher_forcing_ratio=current_meta_tf_ratio
        )
        
        val_loss = evaluate_tf(
            model, val_loader, text_criterion, color_criterion, object_criterion, 
            granger_edge_index, granger_edge_attr, COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
        )
        
        new_scheduler.step(val_loss)
        end_time = time.time()
        formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
        
        print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time} | Meta TF Ratio: {current_meta_tf_ratio:.2f}")
        print(f"\tTrain Loss: {train_loss:.4f}")
        print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print("\t-> Validation loss improved, saving new best Transformer model. 🏆")
        else:
            print("\t-> Validation loss did not improve.")

    print("\n--- Training Complete. ---")


--- 1. Setting up Data and Static Graph ---
Data loaders created: Train=22400, Val=2800, Test=2800


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Static Granger Graph ready on cuda. Edges: 1436

--- 2. Starting Transformer Training (V5) ---
Total Parameters: 23,476,960
Validation uses Honest Evaluation (Meta TF Ratio = 0.0)


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

...


Epoch 08/50 | Time: 05m 01s | Meta TF Ratio: 0.82
	Train Loss: 1.8919
	 Val. Loss: 2.3422 | Val. Perplexity: 10.4044
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 09/50 | Time: 04m 53s | Meta TF Ratio: 0.79
	Train Loss: 1.8072
	 Val. Loss: 2.2581 | Val. Perplexity:  9.5651
	-> Validation loss improved, saving new best Transformer model. 🏆


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
# ==================================================================================
# --- INFERENCE FUNCTION (TRANSFORMER BEAM SEARCH) ---
# ==================================================================================

@torch.no_grad()
def generate_text_beam(model, eeg_signal, meta_signal_gt, edge_index, edge_attr, 
                       beam_width=5, max_len=64):    
    """Generates text using Beam Search for the Transformer Decoder, conditioned
       on the model's own predicted metadata."""
    model.eval()
    
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG (Static Memory)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    decoder_memory = eeg_features[:, 1:, :] 
    memory_mask = None 
    
    # 2. Predict Metadata (Self-Sufficient Conditioning)
    cls_token_feature = eeg_features[:, 0, :]
    meta_preds = model.meta_head(cls_token_feature)
    
    # a) Extract and Process Predicted Logits
    pred_color_logits = meta_preds[:, :model.num_colors]
    pred_object_logits = meta_preds[:, model.num_colors:]

    # b) Create Predicted Metadata Input Vector
    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_soft = torch.sigmoid(pred_object_logits) 
    predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_soft], dim=1)
    
    # c) Encode Predicted Metadata for Gating
    meta_features = model.meta_encoder(predicted_meta_vector)
    
    # 3. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 4. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            input_ids = torch.tensor([seq], dtype=torch.long, device=device)

            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            next_token_logits = logits[:, -1, :].squeeze(0)
            
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob 
                
                new_beams.append((new_seq, new_score))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 5. Final Output Selection
    best_seq = beams[0][0]
    
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    return predicted_text

In [4]:
# ==================================================================================
# --- FINAL EVALUATION SCRIPT (ROUGE & BLEU) ---
# ==================================================================================
if __name__ == "__main__":
    # NOTE: Run this block ONLY after the training loop in the previous code finishes

    # --- Setup ---
    print("\n--- Starting Final BLEU and ROUGE Evaluation ---")
    
    # Load the best weights
    try:
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
        print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
    except Exception as e:
        print(f"Error loading model weights: {e}. Ensure training completed and checkpoint exists.")
        exit()

    predictions_list = []
    references_list = []
    global_sample_index = 0
    
    # Initialize evaluation metrics
    bleu_metric = hf_evaluate.load('bleu')
    rouge_metric = hf_evaluate.load('rouge')

    test_progress_bar = tqdm(test_loader, desc="Beam Search Generation", leave=True)
    start_time = time.time()

    # --- Generation Loop ---
    for eeg_b, meta_b, txt_b in test_progress_bar:
        for i in range(eeg_b.shape[0]):
            eeg_sample = eeg_b[i]
            meta_sample = meta_b[i]
            true_text_ids = txt_b[i]
            
            # Generate text using the Transformer Beam Search
            predicted_text = generate_text_beam(
                model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr, beam_width=5
            )
            
            # --- Ground Truth Text ---
            true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
            
            if not true_text: continue
                
            # Store for Corpus-Level Evaluation
            predictions_list.append(predicted_text)
            # HF 'evaluate' expects a list of reference strings for each sample
            references_list.append(true_text) 

            global_sample_index += 1

    # --- Final Reporting ---
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"

    # Compute BLEU Score
    bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
    final_bleu_score = bleu_results['bleu']

    # Compute ROUGE Score (ROUGE-1, ROUGE-2, ROUGE-L)
    rouge_results = rouge_metric.compute(predictions=predictions_list, references=references_list)

    print("\n=============================================")
    print(f"Evaluation Complete in {formatted_time}")
    print("=============================================")
    print(f"| Total Samples Evaluated:  {global_sample_index} |")
    print("---------------------------------------------")
    print(f"| **Corpus-Level BLEU Score (Beam 5):** {final_bleu_score:.4f} |")
    print("---------------------------------------------")
    print("| ROUGE F1 SCORES: |")
    print(f"| ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f"| ROUGE-2: {rouge_results['rouge2']:.4f}")
    print(f"| ROUGE-L: {rouge_results['rougeL']:.4f}")
    print("=============================================")
    
# This video explores how NLP evaluation metrics like ROUGE and BLEU are calculated and used to assess model performance, which is exactly the final step of your project. https://www.youtube.com/watch?v=7gy-wUAHkBM


--- Starting Final BLEU and ROUGE Evaluation ---
Successfully loaded best model weights from: eeg-meta-text-transformer-v5-model.pt


Beam Search Generation:   0%|          | 0/88 [00:00<?, ?it/s]


Evaluation Complete in 23m 55s
| Total Samples Evaluated:  2800 |
---------------------------------------------
| **Corpus-Level BLEU Score (Beam 5):** 0.0210 |
---------------------------------------------
| ROUGE F1 SCORES: |
| ROUGE-1: 0.2068
| ROUGE-2: 0.0353
| ROUGE-L: 0.1929


In [ ]:
#Refined

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import os
import evaluate as hf_evaluate
from torch.amp import autocast as amp_autocast, GradScaler as amp_GradScaler
import gc

In [6]:
# ------------------------------------------------------------
# 1. CONSTANTS
# ------------------------------------------------------------
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
MODEL_SAVE_PATH = 'eeg-meta-text-transformer-v6-numpy2.pt'
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16
NUM_COLORS = 12
NUM_OBJECTS = 90
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
D_MODEL = 384
D_FF = 2048
NUM_LAYERS = 3
NUM_HEADS = 8
DROPOUT = 0.1
EPOCHS = 60
PRETRAIN_EPOCHS = 5
COLOR_LOSS_WEIGHT = 1.0
OBJECT_LOSS_WEIGHT = 2.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ------------------------------------------------------------
# 2. TRANSFORMER BUILDING BLOCKS (unchanged)
# ------------------------------------------------------------
def get_clones(module, N): return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=DROPOUT):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        if mask is not None: mask = mask.unsqueeze(1)
        bs = q.size(0)
        q, k, v = [l(x).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
                   for l, x in zip(self.linears, (q, k, v))]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        x = torch.matmul(attn, v)
        x = x.transpose(1, 2).contiguous().view(bs, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.meta_gate = nn.Sequential(nn.Linear(d_meta, d_model), nn.Sigmoid())
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)
        self.n3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mem, src_mask, tgt_mask, meta):
        x = self.n1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.n2(x + self.dropout(self.cross_attn(x, mem, mem, src_mask)))
        x = x * self.meta_gate(meta).unsqueeze(1)
        x = self.n3(x + self.dropout(self.ff(x)))
        return x

Device: cuda


In [7]:
# ------------------------------------------------------------
# 3. METADATA ENCODER
# ------------------------------------------------------------
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, c_dim=32, o_dim=256):
        super().__init__()
        self.c_emb = nn.Embedding(num_colors, c_dim)
        self.o_proc = nn.Sequential(
            nn.Linear(num_objects, 512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, o_dim)
        )
        self.output_dim = c_dim + o_dim
    def forward(self, meta):
        c = self.c_emb(meta[:, 0].long())
        o = self.o_proc(meta[:, 1:].float())
        return torch.cat([c, o], dim=1)

# ------------------------------------------------------------
# 4. EEG ENCODER (GCN + Transformer)
# ------------------------------------------------------------
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=D_MODEL, nl=NUM_LAYERS,
                 nh=NUM_HEADS, dff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.pos = PositionalEncoding(d_model)
        layer = TransformerEncoderLayer(d_model, nh, dff, dropout)
        self.layers = get_clones(layer, nl)
        self.norm = nn.LayerNorm(d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        B, C, T = eeg.shape
        # ---- GCN on flattened time ----
        eeg_flat = eeg.permute(0, 2, 1).reshape(-1, C)          # (B*T, C)
        x = F.relu(self.gcn1(eeg_flat, edge_index, edge_attr))
        x = self.drop(x)
        x = F.relu(self.gcn2(x, edge_index, edge_attr))
        x = x.view(B, T, -1)                                    # (B, T, D)
        cls = self.cls.repeat(B, 1, 1)
        x = torch.cat([cls, x], dim=1)                          # (B, T+1, D)
        x = self.pos(x)
        for l in self.layers: x = l(x)
        return self.norm(x)

# ------------------------------------------------------------
# 5. DECODER
# ------------------------------------------------------------
class DecoderTF(nn.Module):
    def __init__(self, vocab, emb_dim, d_model, nl, nh, dff, pad_id, dropout, d_meta):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim, padding_idx=pad_id)
        self.pos = PositionalEncoding(emb_dim)
        self.proj = nn.Linear(emb_dim, d_model)
        layer = TransformerDecoderLayer(d_model, nh, dff, dropout, d_meta)
        self.layers = get_clones(layer, nl)
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab)
    def forward(self, tgt, mem, src_mask, meta):
        x = self.emb(tgt)
        x = self.pos(x)
        x = self.proj(x)
        L = tgt.size(1)
        mask = torch.triu(torch.ones(L, L), diagonal=1).bool().to(x.device)
        mask = mask.unsqueeze(0).unsqueeze(0)
        for l in self.layers: x = l(x, mem, src_mask, mask, meta)
        return self.out(self.norm(x))

# ------------------------------------------------------------
# 6. FULL SEQ2SEQ MODEL
# ------------------------------------------------------------
class Seq2SeqTF(nn.Module):
    def __init__(self, vocab, nc, no):
        super().__init__()
        self.enc = SpatioTemporalEEGEncoderTF()
        self.meta_enc = MetadataEncoder(nc, no)
        d_meta = self.meta_enc.output_dim
        self.dec = DecoderTF(vocab, D_MODEL, D_MODEL, NUM_LAYERS, NUM_HEADS,
                             D_FF, PAD_ID, DROPOUT, d_meta)
        # stronger meta head
        self.meta_head = nn.Sequential(
            nn.Linear(D_MODEL, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, nc + no)
        )
        self.nc = nc
        self.no = no

    def forward(self, eeg, meta, tgt, edge_idx, edge_w, tf_ratio=1.0):
        enc = self.enc(eeg, edge_idx, edge_w)               # (B, T+1, D)
        cls = enc[:, 0]
        meta_pred = self.meta_head(cls)

        use_gt = random.random() < tf_ratio
        if use_gt:
            mfeat = self.meta_enc(meta)
        else:
            c_id = meta_pred[:, :self.nc].argmax(dim=-1).float().unsqueeze(1)
            o_soft = torch.sigmoid(meta_pred[:, self.nc:])
            mfeat = self.meta_enc(torch.cat([c_id, o_soft], dim=1))

        var = mfeat.var(dim=0).mean()
        var_pen = 0.001 / (var + 1e-6)

        mem = enc[:, 1:, :]
        logits = self.dec(tgt[:, :-1], mem, None, mfeat)

        return logits, meta_pred[:, :self.nc], meta_pred[:, self.nc:], var_pen

# ------------------------------------------------------------
# 7. DATASET
# ------------------------------------------------------------
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, path):
        self.path = path
        with h5py.File(path, 'r') as f: self.N = f['eeg'].shape[0]
        self.f = None
    def __len__(self): return self.N
    def __getitem__(self, i):
        if self.f is None: self.f = h5py.File(self.path, 'r')
        return (torch.from_numpy(self.f['eeg'][i].astype('float32')),
                torch.from_numpy(self.f['metadata'][i].astype('float32')),
                torch.from_numpy(self.f['input_ids'][i].astype('int64')))

def collate(batch):
    e, m, t = zip(*batch)
    return torch.stack(e), torch.stack(m), pad_sequence(t, batch_first=True, padding_value=PAD_ID)

# ------------------------------------------------------------
# 8. FULLY-CONNECTED GRAPH (no scipy)
# ------------------------------------------------------------
def fully_connected_graph(num_nodes):
    """Returns edge_index (2, E) and edge_attr (E,) for a dense graph (no self-loops)."""
    rows = []
    cols = []
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i == j: continue
            rows.append(i)
            cols.append(j)
    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    edge_attr = torch.ones(edge_index.size(1), dtype=torch.float)
    return edge_index, edge_attr

# ------------------------------------------------------------
# 9. TORCH-ONLY mAP (replaces sklearn)
# ------------------------------------------------------------
def torch_average_precision(pred, target, eps=1e-7):
    """pred: (B, K) logits, target: (B, K) binary."""
    prob = torch.sigmoid(pred)
    # sort by descending probability
    order = prob.argsort(dim=1, descending=True)
    target = target.gather(1, order)
    prob = prob.gather(1, order)
    tp = target.cumsum(dim=1).float()
    fp = (1 - target).cumsum(dim=1).float()
    recall = tp / (target.sum(dim=1, keepdim=True) + eps)
    precision = tp / (tp + fp + eps)
    # integrate
    ap = (precision[:, 1:] * (recall[:, 1:] - recall[:, :-1])).sum(dim=1)
    return ap.mean().item()

# ------------------------------------------------------------
# 10. INFERENCE (nucleus sampling)
# ------------------------------------------------------------
@torch.no_grad()
def generate(model, eeg, edge_idx, edge_w, tokenizer, max_len=50, temp=0.8, top_p=0.9):
    model.eval()
    eeg = eeg.unsqueeze(0).to(device)
    ids = torch.tensor([[SOS_ID]], device=device)
    for _ in range(max_len):
        logits, _, _, _ = model(eeg, None, ids, edge_idx, edge_w, tf_ratio=0.0)
        nxt = logits[0, -1] / temp
        # top-p
        sorted_l, sorted_i = torch.sort(nxt, descending=True)
        cum = torch.cumsum(F.softmax(sorted_l, dim=-1), dim=-1)
        mask = cum > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = 0
        nxt[sorted_i[mask]] = -float('inf')
        probs = F.softmax(nxt, dim=-1)
        nxt_id = torch.multinomial(probs, 1).item()
        if nxt_id in (EOS_ID, PAD_ID): break
        ids = torch.cat([ids, torch.tensor([[nxt_id]], device=device)], dim=1)
    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)

In [8]:
# ------------------------------------------------------------
# 11. TRAINING / EVAL
# ------------------------------------------------------------
def train_one(model, loader, opt, tc, cc, oc, ei, ew, cw, ow, tf):
    model.train()
    tot = 0.0
    for e, m, t in tqdm(loader, desc="train", leave=False):
        e, m, t = e.to(device), m.to(device), t.to(device)
        opt.zero_grad()
        logits, pc, po, vp = model(e, m, t, ei, ew, tf)
        loss_t = tc(logits.reshape(-1, logits.size(-1)), t[:, 1:].reshape(-1))
        loss_c = cc(pc, m[:, 0].long())
        loss_o = oc(po, m[:, 1:].float())
        loss = loss_t + cw*loss_c + ow*loss_o + vp
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    return tot / len(loader)

@torch.no_grad()
def evaluate(model, loader, tc, cc, oc, ei, ew, cw, ow):
    model.eval()
    tot = 0.0
    c_corr = 0
    N = 0
    all_po, all_mo = [], []
    for e, m, t in loader:
        e, m, t = e.to(device), m.to(device), t.to(device)
        logits, pc, po, _ = model(e, m, t, ei, ew, 0.0)
        loss_t = tc(logits.reshape(-1, logits.size(-1)), t[:, 1:].reshape(-1))
        loss_c = cc(pc, m[:, 0].long())
        loss_o = oc(po, m[:, 1:].float())
        loss = loss_t + cw*loss_c + ow*loss_o
        tot += loss.item()
        c_corr += (pc.argmax(dim=1) == m[:, 0].long()).sum().item()
        N += m.size(0)
        all_po.append(po)
        all_mo.append(m[:, 1:])
    acc = c_corr / N
    po = torch.cat(all_po); mo = torch.cat(all_mo)
    mAP = torch_average_precision(po, mo)
    return tot / len(loader), acc, mAP

In [19]:
# ------------------------------------------------------------
# 12. MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    # ---- data ----
    ds = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(ds)
    trN = int(N*TRAIN_PCT); valN = int(N*VAL_PCT); teN = N-trN-valN
    tr, va, te = random_split(ds, [trN, valN, teN], generator=torch.Generator().manual_seed(42))
    trL = DataLoader(tr, BATCH_SIZE, shuffle=True, collate_fn=collate)
    vaL = DataLoader(va, BATCH_SIZE, shuffle=False, collate_fn=collate)
    teL = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate)

    # ---- static fully-connected graph (62 channels) ----
    edge_idx, edge_w = fully_connected_graph(62)
    edge_idx = edge_idx.to(device)
    edge_w = edge_w.to(device)

    # ---- model & losses ----
    model = Seq2SeqTF(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS).to(device)

    # === CLEAR GPU MEMORY ===
    print("Clearing GPU...")
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

    tc = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
    cc = nn.CrossEntropyLoss()
    oc = nn.BCEWithLogitsLoss()

    # ------------------------------------------------------------
    # ---- PRE-TRAIN META HEAD (FIXED & SAFE) ----
    # ------------------------------------------------------------
    print("\nPre-training meta head (decoder frozen)...")

    for p in model.dec.parameters():
        p.requires_grad = False

    PRETRAIN_BATCH_SIZE = 16
    ACCUM_STEPS = 4
    pretrain_loader = DataLoader(tr, PRETRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate)

    opt_pre = AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)
    scaler = amp_GradScaler('cuda')

    def torch_ap(pl, t):
        p = torch.sigmoid(pl)
        o = p.argsort(dim=1, descending=True)
        t = t.gather(1, o); p = p.gather(1, o)
        tp = t.cumsum(1).float(); fp = (1-t).cumsum(1).float()
        r = tp / (t.sum(1, keepdim=True) + 1e-7)
        pr = tp / (tp + fp + 1e-7)
        return (pr[:, 1:] * (r[:, 1:] - r[:, :-1])).sum(1).mean().item()

    for epoch in range(PRETRAIN_EPOCHS):
        epoch_loss = 0.0
        prog = tqdm(pretrain_loader, desc=f"Pre-train {epoch+1}/{PRETRAIN_EPOCHS}", leave=False)

        opt_pre.zero_grad()

        for step, (e, m, _) in enumerate(prog):
            e, m = e.to(device), m.to(device)

            with amp_autocast('cuda'):
                enc = model.enc(e, edge_idx, edge_w)
                cls = enc[:, 0]
                mp = model.meta_head(cls)
                loss = (cc(mp[:, :12], m[:, 0].long()) + 2.0 * oc(mp[:, 12:], m[:, 1:].float())) / ACCUM_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(pretrain_loader):
                scaler.unscale_(opt_pre)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt_pre)
                scaler.update()
                opt_pre.zero_grad()

            epoch_loss += loss.item() * ACCUM_STEPS
            prog.set_postfix(loss=f"{loss.item()*ACCUM_STEPS:.3f}")

        # ---- METRICS ----
        model.eval()
        with torch.no_grad():
            ev, mv, _ = next(iter(pretrain_loader))
            ev, mv = ev.to(device), mv.to(device)
            enc_v = model.enc(ev, edge_idx, edge_w)
            cls_v = enc_v[:, 0]
            pred_v = model.meta_head(cls_v)
            cacc = (pred_v[:, :12].argmax(1) == mv[:, 0].long()).float().mean().item()
            oap = torch_ap(pred_v[:, 12:], mv[:, 1:].float())
        model.train()

        print(f"→ Epoch {epoch+1} | Loss: {epoch_loss/len(pretrain_loader):.4f} | "
              f"ColorAcc: {cacc:.3f} | Obj mAP: {oap:.3f}")

    # Unfreeze
    for p in model.dec.parameters():
        p.requires_grad = True

    print("Pre-training DONE. Starting full training...\n")

    # ------------------------------------------------------------
    # ---- full training ----
    # ------------------------------------------------------------
    opt = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    sched = ReduceLROnPlateau(opt, 'min', factor=0.5, patience=3, verbose=True)
    tf_sched = np.concatenate([np.ones(30), np.linspace(1.0, 0.0, 30)])
    best = float('inf')

    for ep in range(1, EPOCHS+1):
        tf = tf_sched[min(ep-1, len(tf_sched)-1)]
        tr_loss = train_one(model, trL, opt, tc, cc, oc, edge_idx, edge_w,
                            COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT, tf)
        val_loss, cacc, omap = evaluate(model, vaL, tc, cc, oc, edge_idx, edge_w,
                                        COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT)
        sched.step(val_loss)
        print(f"Ep {ep:02d} | TF {tf:.2f} | Tr {tr_loss:.4f} | Val {val_loss:.4f} | "
              f"ColorAcc {cacc:.3f} | ObjAP {omap:.3f}")
        if val_loss < best:
            best = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(" -> saved best")

Clearing GPU...
GPU Memory: 0.72 GB allocated

Pre-training meta head (decoder frozen)...


Pre-train 1/5:   0%|          | 0/1400 [00:00<?, ?it/s]

→ Epoch 1 | Loss: 2.5024 | ColorAcc: 0.188 | Obj mAP: 0.141


Pre-train 2/5:   0%|          | 0/1400 [00:00<?, ?it/s]

→ Epoch 2 | Loss: 2.4154 | ColorAcc: 0.125 | Obj mAP: 0.130


...

Ep 05 | TF 1.00 | Tr 4.5245 | Val 5.6063 | ColorAcc 0.239 | ObjAP 0.138


train:   0%|          | 0/1400 [00:00<?, ?it/s]

Ep 06 | TF 1.00 | Tr 4.3605 | Val 5.6668 | ColorAcc 0.239 | ObjAP 0.138


train:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
import torch
def print_gpu_memory():
    if torch.cuda.is_available():
        print(f"GPU Memory: "
              f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | "
              f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB | "
              f"Max:       {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Call it like this:
print_gpu_memory()

GPU Memory: Allocated: 0.00 GB | Reserved:  0.00 GB | Max:       0.00 GB


In [21]:
# ------------------------------------------------------------
#  EVALUATION ON TEST SET (after training)
# ------------------------------------------------------------
print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# -------------------------------------------------
# Load the best checkpoint
# -------------------------------------------------
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()
print(f"Loaded best model from: {MODEL_SAVE_PATH}")

# -------------------------------------------------
# Test loader (same split you used for training)
# -------------------------------------------------
test_loader = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate)

# -------------------------------------------------
# 1. Full test-set metrics (loss + ColorAcc + ObjAP)
# -------------------------------------------------
test_loss, test_cacc, test_omap = evaluate(
    model, test_loader, tc, cc, oc, edge_idx, edge_w,
    COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
)

print("\n=== TEST METRICS ===")
print(f"Test loss : {test_loss:.4f}")
print(f"ColorAcc  : {test_cacc:.3f}")
print(f"Obj mAP   : {test_omap:.3f}")

# -------------------------------------------------
# 2. Generate a few concrete examples
# -------------------------------------------------
print("\n=== SAMPLE GENERATIONS ===")
with torch.no_grad():
    for batch_idx, (eeg_batch, meta_batch, txt_batch) in enumerate(test_loader):
        if batch_idx >= 3:                     # show only first 3 batches
            break
        eeg_batch = eeg_batch.to(device)

        for j in range(min(2, eeg_batch.shape[0])):   # 2 examples per batch
            eeg = eeg_batch[j:j+1]                    # keep batch dim = 1

            # ---- generate ----
            pred_ids = generate(
                model, eeg, edge_idx, edge_w, tokenizer,
                max_len=20, temp=1.0, top_p=0.9
            )
            pred_txt = tokenizer.decode(pred_ids, skip_special_tokens=True)
            true_txt = tokenizer.decode(txt_batch[j].tolist(),
                                       skip_special_tokens=True)

            # ---- meta prediction (for fun) ----
            enc = model.enc(eeg, edge_idx, edge_w)
            cls = enc[:, 0]
            mp  = model.meta_head(cls)
            pred_color = mp[:, :12].argmax(1).item()
            true_color = meta_batch[j, 0].item()

            print(f"\n--- Example {batch_idx*2 + j + 1} ---")
            print(f"EEG → Predicted : {pred_txt}")
            print(f"      Ground    : {true_txt}")
            print(f"      Color pred: {pred_color} | true: {true_color}")

# -------------------------------------------------
# 3. (Optional) Full-test BLEU – fast version on first 200 samples
# -------------------------------------------------
print("\n=== BLEU (first 200 samples) ===")
import sacrebleu
hyps, refs = [], []
model.eval()
with torch.no_grad():
    for eeg_batch, _, txt_batch in test_loader:
        eeg_batch = eeg_batch.to(device)
        for j in range(eeg_batch.shape[0]):
            pred_ids = generate(
                model, eeg_batch[j:j+1], edge_idx, edge_w,
                tokenizer, max_len=20, temp=1.0, top_p=0.9
            )
            hyps.append(tokenizer.decode(pred_ids, skip_special_tokens=True))
            refs.append([tokenizer.decode(txt_batch[j].tolist(),
                                         skip_special_tokens=True)])
            if len(hyps) >= 200:
                break
        if len(hyps) >= 200:
            break

bleu = sacrebleu.corpus_bleu(hyps, refs)
print(f"BLEU : {bleu.score:.2f}")

print("\nEvaluation finished!")


EVALUATION ON TEST SET
Loaded best model from: eeg-meta-text-transformer-v6-numpy2.pt

=== TEST METRICS ===
Test loss : 5.5438
ColorAcc  : 0.253
Obj mAP   : 0.142

=== SAMPLE GENERATIONS ===


ValueError: too many values to unpack (expected 3)